In [ ]:
# 1/8 - GPU + deps (real GPU timing on both sides)
import os
os.environ['CUDA_LAUNCH_BLOCKING']='1'
!nvidia-smi
import sys
print(sys.version)
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
!pip -q install transformers

In [ ]:
# 2/8 - Fresh clone TokenVector (bridge source)
import subprocess
d='/content/TokenVector'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 3/8 - Build tkv_torch_fused.so (ATen bridge)
import subprocess, os, sys, ctypes
os.environ['TKV_TORCH_GPU']='1'
torch_dir=subprocess.check_output([sys.executable,'-c','import torch,os; print(os.path.dirname(torch.__file__))']).decode().strip()
incs=[torch_dir+'/include', torch_dir+'/include/torch/csrc/api/include']
libdir=torch_dir+'/lib'
cmd=('cd /content/TokenVector/Release/tools/simd_native && '
     'g++ -O2 -shared -std=c++20 -fPIC torch_fused.cpp -o tkv_torch_fused.so '
     '-I'+incs[0]+' -I'+incs[1]+' -L'+libdir+' '
     '-Wl,-rpath,'+libdir+' -ltorch -ltorch_cpu -lc10 -lc10_cuda -ltorch_cuda')
p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
print('build rc =', p.returncode)
if p.returncode!=0: print('STDERR:', p.stderr[:6000])
so='/content/TokenVector/Release/tools/simd_native/tkv_torch_fused.so'
os.environ['TKV_TORCH_PATH']=so
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
lib=ctypes.CDLL(so)
print('ctypes load OK', os.path.exists(so))

In [ ]:
# 4/8 - Data + pretrained GPT2-medium + identical batches
import numpy as np, torch, urllib.request, copy, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, GPT2Config
T=512; V=50257; NL=24; H=1024; NH=16; HD=64; STEPS=400
tok=AutoTokenizer.from_pretrained('gpt2')
url='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text=urllib.request.urlopen(url, timeout=60).read().decode('utf-8')
ids_all=np.array(tok.encode(text), dtype=np.int64)
print('corpus chars=',len(text),' tokens=',len(ids_all))
split=int(len(ids_all)*0.9); train_ids=ids_all[:split]; val_ids=ids_all[split:]
print('train/val tokens=',len(train_ids),len(val_ids))
cfg=GPT2Config(n_layer=NL,n_head=NH,n_embd=H,n_positions=1024,vocab_size=V,
               layer_norm_epsilon=1e-6, activation_function='gelu',
               resid_pdrop=0.0, attn_pdrop=0.0, embd_pdrop=0.0)
model=AutoModelForCausalLM.from_pretrained('gpt2-medium', config=cfg).eval()
sd=model.state_dict()
sd_initial={k:v.detach().clone() for k,v in sd.items()}
def blocks():
    out=[sd['transformer.wte.weight'].numpy(), sd['transformer.wpe.weight'].numpy()[:T],
         sd['transformer.ln_f.weight'].numpy(), sd['transformer.ln_f.bias'].numpy()]
    for i in range(NL):
        p='transformer.h.%d.'%i
        out += [sd[p+'ln_1.weight'].numpy(), sd[p+'ln_1.bias'].numpy(),
                sd[p+'attn.c_attn.weight'].numpy().T.copy(), sd[p+'attn.c_attn.bias'].numpy(),
                sd[p+'attn.c_proj.weight'].numpy().T.copy(), sd[p+'attn.c_proj.bias'].numpy(),
                sd[p+'ln_2.weight'].numpy(), sd[p+'ln_2.bias'].numpy(),
                sd[p+'mlp.c_fc.weight'].numpy().T.copy(), sd[p+'mlp.c_fc.bias'].numpy(),
                sd[p+'mlp.c_proj.weight'].numpy().T.copy(), sd[p+'mlp.c_proj.bias'].numpy()]
    return out
params=blocks()
flat=np.concatenate([x.reshape(-1) for x in params]).astype(np.float32)
assert len(params)==4+NL*12
rng=np.random.default_rng(7)
batches=[]
for _ in range(STEPS):
    s=int(rng.integers(0,len(train_ids)-T-1))
    batches.append((train_ids[s:s+T].copy(), train_ids[s+1:s+T+1].copy()))
print('tensors=',len(params),' params=',flat.size,' batches=',len(batches))
def flat_to_state(arr, template):
    d=dict(template); off=0
    def take(shape):
        nonlocal off
        z=np.asarray(arr[off:off+int(np.prod(shape))],dtype=np.float32).reshape(shape); off+=z.size
        return torch.from_numpy(z.copy())
    d['transformer.wte.weight']=take((V,H))
    wpe=torch.zeros(1024,H,dtype=torch.float32); wpe[:T]=take((T,H)); d['transformer.wpe.weight']=wpe
    d['transformer.ln_f.weight']=take((H,)); d['transformer.ln_f.bias']=take((H,))
    for i in range(NL):
        p='transformer.h.%d.'%i
        d[p+'ln_1.weight']=take((H,)); d[p+'ln_1.bias']=take((H,))
        d[p+'attn.c_attn.weight']=take((3*H,H)).T.contiguous(); d[p+'attn.c_attn.bias']=take((3*H,))
        d[p+'attn.c_proj.weight']=take((H,H)).T.contiguous(); d[p+'attn.c_proj.bias']=take((H,))
        d[p+'ln_2.weight']=take((H,)); d[p+'ln_2.bias']=take((H,))
        d[p+'mlp.c_fc.weight']=take((4*H,H)).T.contiguous(); d[p+'mlp.c_fc.bias']=take((4*H,))
        d[p+'mlp.c_proj.weight']=take((H,4*H)).T.contiguous(); d[p+'mlp.c_proj.bias']=take((H,))
    print('consumed',off,'of',arr.size)
    return d



In [ ]:
# 5/8 - PyTorch eager reference (same model/data, real GPU time)
import time, statistics as st
ref_model=copy.deepcopy(model).train().to('cuda')
ref_opt=torch.optim.AdamW(ref_model.parameters(), lr=5e-5, betas=(0.9,0.999), eps=1e-8, weight_decay=0.0)
ref_losses=[]; ref_times=[]
for step,(xb,yb) in enumerate(batches):
    xi=torch.from_numpy(xb.astype(np.int64)).unsqueeze(0).cuda()
    yi=torch.from_numpy(yb.astype(np.int64)).unsqueeze(0).cuda()
    t0=time.perf_counter()
    loss=ref_model(input_ids=xi, labels=xi).loss
    loss.backward(); ref_opt.step(); ref_opt.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    ref_times.append((time.perf_counter()-t0)*1000); ref_losses.append(float(loss))
    if step%50==0 or step==STEPS-1: print('torch step %4d loss %.4f  %.1f ms'%(step,loss,ref_times[-1]))
print('TORCH median(ms) last50=%.1f  loss %.4f -> %.4f'%(st.median(ref_times[-50:]),ref_losses[0],ref_losses[-1]))
torch_ref_losses=list(ref_losses)
del ref_model, ref_opt, xi, yi; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 6/8 - Register params + optimizer in native fullstep bridge
F,I32,I64=ctypes.c_float, ctypes.c_int32, ctypes.c_int64
lib.TkvFullBegin.argtypes=[I64]*6; lib.TkvFullBegin.restype=ctypes.c_int
lib.TkvFullSetTensor.argtypes=[I64,ctypes.POINTER(F),I64]; lib.TkvFullSetTensor.restype=ctypes.c_int
lib.TkvFullSetIds.argtypes=[ctypes.POINTER(I32),I64]; lib.TkvFullSetIds.restype=ctypes.c_int
lib.TkvFullSetTargets.argtypes=[ctypes.POINTER(I32),I64]; lib.TkvFullSetTargets.restype=ctypes.c_int
lib.TkvFullSetGrad.argtypes=[ctypes.POINTER(F),I64]; lib.TkvFullSetGrad.restype=ctypes.c_int
lib.TkvFullSetOpt.argtypes=[ctypes.POINTER(F)]*3+[F]*5+[I64]*2; lib.TkvFullSetOpt.restype=ctypes.c_int
lib.TkvFullStep124m.argtypes=[I64]; lib.TkvFullStep124m.restype=F
lib.TkvFullGetParams.argtypes=[ctypes.POINTER(F),I64]; lib.TkvFullGetParams.restype=ctypes.c_int
n=flat.size
grad=np.zeros(n,dtype=np.float32); m=np.zeros(n,dtype=np.float32); v=np.zeros(n,dtype=np.float32)
print('BEGIN', lib.TkvFullBegin(len(params),NL,H,NH,HD,V))
for i,a in enumerate(params):
    aa=np.ascontiguousarray(a,dtype=np.float32)
    r=lib.TkvFullSetTensor(i, aa.ctypes.data_as(ctypes.POINTER(F)), aa.size)
    if r!=0: print('SET_FAIL',i,r)
xb0,yb0=batches[0]
print('IDS', lib.TkvFullSetIds(xb0.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T))
print('TGT', lib.TkvFullSetTargets(yb0.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T))
print('GRAD', lib.TkvFullSetGrad(grad.ctypes.data_as(ctypes.POINTER(F)),n))
print('OPT', lib.TkvFullSetOpt(flat.ctypes.data_as(ctypes.POINTER(F)),m.ctypes.data_as(ctypes.POINTER(F)),v.ctypes.data_as(ctypes.POINTER(F)),5e-5,0.9,0.999,1e-8,0.0,STEPS,0))

In [ ]:
# diag - isolate fp16 vs layout: HF fp32 / HF fp16 / TKV forward on SAME batch
import copy, torch
xb0,yb0=batches[0]
xi=torch.from_numpy(xb0.astype(np.int64)).unsqueeze(0)
with torch.no_grad():
    l32=float(model(input_ids=xi, labels=xi).loss)
m16=copy.deepcopy(model).half().to('cuda').eval()
with torch.no_grad():
    l16=float(m16(input_ids=xi.cuda(), labels=xi.cuda()).loss)
del m16; torch.cuda.empty_cache()
ltk=float(lib.TkvFullStep124m(1))   # forward only, does NOT update weights
print('HF fp32 loss=%.4f  HF fp16 loss=%.4f  TKV forward loss=%.4f'%(l32,l16,ltk))
print('if HF fp16 ~ TKV -> fp16 numerics; if HF fp16 ~ HF fp32 -> layout/code bug')
# decisive: roundtrip initial flat params back into HF (mapping check)
m0=copy.deepcopy(model).cpu()
m0.load_state_dict(flat_to_state(flat,sd))
with torch.no_grad():
    l_rt=float(m0(input_ids=xi, labels=xi).loss)
del m0; torch.cuda.empty_cache()
print('HF roundtrip(flat)->loss=%.4f  (should ~= HF fp32 %.4f)'%(l_rt,l32))

In [ ]:
# 7/8 - Native fullstep training + head-to-head benchmark
tkv_losses=[]; tkv_times=[]; t0=time.perf_counter()
for step,(xb,yb) in enumerate(batches):
    lib.TkvFullSetIds(xb.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T)
    lib.TkvFullSetTargets(yb.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T)
    t1=time.perf_counter()
    loss=float(lib.TkvFullStep124m(0))
    tkv_times.append((time.perf_counter()-t1)*1000); tkv_losses.append(loss)
    if step%50==0 or step==STEPS-1: print('tkv  step %4d loss %.4f  %.1f ms'%(step,loss,tkv_times[-1]))
print('TKV  median(ms) last50=%.1f  loss %.4f -> %.4f'%(st.median(tkv_times[-50:]),tkv_losses[0],tkv_losses[-1]))
print('SPEEDUP vs torch = %.2fx'%(st.median(ref_times[-50:])/st.median(tkv_times[-50:])))
print('first-step loss: torch %.4f / tkv %.4f (close = layout OK)'%(torch_ref_losses[0],tkv_losses[0]))

In [ ]:
# 8/8 - Save fp32 checkpoint + generate real sample
trained=np.zeros(n,dtype=np.float32)
print('GET rc=', lib.TkvFullGetParams(trained.ctypes.data_as(ctypes.POINTER(F)),n))
np.save('/content/gpt2medium_tiny_shakespeare_fp32.npy',trained)
import json
json.dump({'n_layer':NL,'n_head':NH,'n_embd':H,'n_positions':T,'vocab_size':V,'corpus':'tiny_shakespeare','steps':STEPS}, open('/content/gpt2medium_train_config.json','w'))
model.load_state_dict(flat_to_state(trained,sd))
model.to('cuda').eval()
try:
    prompt='ROMEO: '
    ids=tok(prompt,return_tensors='pt').input_ids.cuda()
    with torch.no_grad():
        out=model.generate(ids,max_new_tokens=200,do_sample=True,temperature=0.8,top_k=40)
    print('--- SAMPLE ---')
    print(tok.decode(out[0]))
except Exception as e:
    print('GENERATE SKIPPED:', type(e).__name__, str(e)[:300])

In [ ]:
# 9/9 - TEST: held-out perplexity (before/after) + continuation sample
import math, copy, numpy as np, torch
try:
    trained
except NameError:
    trained=np.load('/content/gpt2medium_tiny_shakespeare_fp32.npy')
model.load_state_dict(flat_to_state(trained,sd))
val_windows=[]
rngv=np.random.default_rng(99)
for _ in range(30):
    s=int(rngv.integers(0,len(val_ids)-T-1))
    val_windows.append((val_ids[s:s+T].copy(), val_ids[s+1:s+T+1].copy()))
def ppl_hf(state, tag):
    m=copy.deepcopy(model).cpu(); m.load_state_dict(state); m.to('cuda').eval()
    ls=[]
    with torch.no_grad():
        for xb,yb in val_windows:
            xi=torch.from_numpy(xb.astype(np.int64)).unsqueeze(0).cuda()
            yi=torch.from_numpy(yb.astype(np.int64)).unsqueeze(0).cuda()
            ls.append(float(m(input_ids=xi,labels=xi).loss))
    del m; torch.cuda.empty_cache()
    p=math.exp(float(np.mean(ls)))
    print('%-16s loss=%.4f  perplexity=%.2f'%(tag,np.mean(ls),p))
    return p
ppl_before=ppl_hf(sd_initial,'pretrained')
ppl_after=ppl_hf(model.state_dict(),'fine-tuned(HF)')
# same test through the native bridge (forward-only, run_mode=1 -> no optimizer step)
bl=[]
for xb,yb in val_windows:
    lib.TkvFullSetIds(xb.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T)
    lib.TkvFullSetTargets(yb.astype(np.int32).ctypes.data_as(ctypes.POINTER(I32)),T)
    bl.append(float(lib.TkvFullStep124m(1)))
ppl_bridge=math.exp(float(np.mean(bl)))
print('%-16s loss=%.4f  perplexity=%.2f'%('fine-tuned(TKV)',np.mean(bl),ppl_bridge))
print('IMPROVEMENT: perplexity %.2f -> %.2f (%.1f%% better)'%(ppl_before,ppl_after,100*(1-ppl_after/ppl_before)))
# qualitative: continue a HELD-OUT prompt
seed=tok.decode(val_ids[:T])[-160:]
ids=tok(seed,return_tensors='pt').input_ids.cuda()
with torch.no_grad():
    out=model.generate(ids,max_new_tokens=160,do_sample=True,temperature=0.8,top_k=40)
print('--- HELD-OUT CONTINUATION ---')
print(tok.decode(out[0]))